# Kaggle - 5 LCTA resize methods with mBERT and mT5

ATE trains once with `google/mt5-large`. APC then trains 5 methods with mBERT and 5 methods with mT5, for 10 APC runs total.

In [ ]:
import os
import sys
import subprocess
import zipfile
from pathlib import Path

REPO_URL = 'https://github.com/hotuyen21pt/KLTN-Token-Merging.git'
BRANCH = 'tuyen'
REPO_NAME = 'KLTN-Token-Merging'
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
REPO_DIR = WORK_ROOT / REPO_NAME

# Kaggle Input: point this to the uploaded zip containing train.apc, dev.apc, test.apc.
DATA_INPUT_ZIP = None
# Optional alternative when the files are already extracted.
DATA_INPUT_DIR = None
# Optional precomputed ATE CSV. Leave None to train mT5-large ATE below.
ATE_INPUT_CSV = None

SEED = 42
MAX_EPOCHS = 15
ATE_EPOCHS = 20
ATE_MODEL_NAME = 'google/mt5-large'
APC_MODELS = {
    'mbert': 'bert-base-multilingual-cased',
    'mt5': 'google/mt5-large',
}
RUNS_DIR = REPO_DIR / 'runs_mbert_mt5_5_methods'
METHODS = {
    'lcf_bip_cdm': 'LCTA-BiToMe-CDM',
    'lcf_seq_cdm': 'LCTA-SLM-CDM',
    'lcf_seq_cdw': 'LCTA-SLM-CDW',
    'lcf_scm_cdm': 'LCTA-SCM-CDM',
    'lcf_scm_cdw': 'LCTA-SCM-CDW',
}
CACHE_DIR = Path('/kaggle/temp/hf') if Path('/kaggle').exists() else WORK_ROOT / '.hf'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'
print('Zip input:', DATA_INPUT_ZIP or '(none)')
print('Directory input:', DATA_INPUT_DIR or '(none)')
print('ATE model:', ATE_MODEL_NAME)
print('APC models:', APC_MODELS)
print('APC runs:', len(METHODS) * len(APC_MODELS))

In [ ]:
def sh(*args, cwd=None, check=True):
    cmd = [str(x) for x in args]
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check).returncode

if (REPO_DIR / '.git').is_dir():
    sh('git', 'fetch', '--all', '--prune', cwd=REPO_DIR)
    sh('git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}', cwd=REPO_DIR)
    sh('git', 'reset', '--hard', f'origin/{BRANCH}', cwd=REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh('git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sh(sys.executable, '-m', 'pip', 'install', '-q', 'python-Levenshtein>=0.25.0', 'seaborn', 'tqdm')

In [ ]:
# Download all model files before training.
# mT5-large is used by ATE and APC; mBERT is used by APC.
from huggingface_hub import snapshot_download

for model_name in sorted(set(APC_MODELS.values()) | {ATE_MODEL_NAME}):
    print('Downloading complete snapshot:', model_name)
    snapshot_path = snapshot_download(
        repo_id=model_name,
        cache_dir=CACHE_DIR,
        resume_download=True,
    )
    print('Ready:', snapshot_path)
print('All required model files are cached.')

In [ ]:
# Resolve data paths. A zip may contain the APC files at its root or one nested folder.
if DATA_INPUT_ZIP:
    zip_path = Path(DATA_INPUT_ZIP)
    if not zip_path.is_file():
        raise FileNotFoundError(f'DATA_INPUT_ZIP does not exist: {zip_path}')
    extracted_root = WORK_ROOT / 'prism_dataset_apc_input'
    extracted_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extracted_root)
    candidates = [
        p.parent for p in extracted_root.rglob('train.apc')
        if (p.parent / 'dev.apc').is_file() and (p.parent / 'test.apc').is_file()
    ]
    if not candidates:
        raise FileNotFoundError(
            'The zip must contain train.apc, dev.apc and test.apc '
            'in the same directory.'
        )
    DATA_DIR = candidates[0]
elif DATA_INPUT_DIR:
    DATA_DIR = Path(DATA_INPUT_DIR)
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f'DATA_INPUT_DIR does not exist: {DATA_DIR}')
else:
    DATA_DIR = REPO_DIR / 'dataset'

required = [DATA_DIR / name for name in ('train.apc', 'dev.apc', 'test.apc')]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing dataset files:\n' + '\n'.join(missing))

ATE_CSV = Path(ATE_INPUT_CSV) if ATE_INPUT_CSV else REPO_DIR / 'runs_ate' / 'test_ate_predictions.csv'
print('Using dataset:', DATA_DIR)
print('ATE CSV:', ATE_CSV if ATE_CSV.is_file() else 'not found (ATE cell will create it)')

In [ ]:
# Export the three APC files as Kaggle output before training.
import shutil

APC_OUTPUT_DIR = WORK_ROOT / 'apc_output'
APC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for filename in ('train.apc', 'dev.apc', 'test.apc'):
    shutil.copy2(DATA_DIR / filename, APC_OUTPUT_DIR / filename)

print('APC output files:')
for filename in ('train.apc', 'dev.apc', 'test.apc'):
    output_file = APC_OUTPUT_DIR / filename
    print(f'  {output_file} ({output_file.stat().st_size / 1024:.1f} KB)')

apc_zip = WORK_ROOT / 'apc_output.zip'
with zipfile.ZipFile(apc_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for filename in ('train.apc', 'dev.apc', 'test.apc'):
        archive.write(APC_OUTPUT_DIR / filename, filename)
print('APC zip output:', apc_zip)

In [ ]:
# Full pipeline stage 1: train mT5-large ATE once and predict aspect terms on test.
import common.run_multiseed_ate as ate_runner

ATE_RUNS_DIR = REPO_DIR / 'runs_ate_mt5_large_pipeline'
ATE_CKPT_DIR = REPO_DIR / 'checkpoints' / 'gas_mt5_large_ate_pipeline'

if ATE_INPUT_CSV:
    ATE_CSV = Path(ATE_INPUT_CSV)
    if not ATE_CSV.is_file():
        raise FileNotFoundError(f'ATE_INPUT_CSV does not exist: {ATE_CSV}')
else:
    ate_args = [
        '--seeds', str(SEED),
        '--model-name', ATE_MODEL_NAME,
        '--epochs', str(ATE_EPOCHS),
        '--lr', '3e-4',
        '--data-dir', str(DATA_DIR),
        '--runs-ate-dir', str(ATE_RUNS_DIR),
        '--ckpt-dir', str(ATE_CKPT_DIR),
        '--resume',
    ]
    print('ATE command: python common/run_multiseed_ate.py ' + ' '.join(ate_args))
    sys.argv = ['run_multiseed_ate.py', *ate_args]
    ate_runner.main()
    ATE_CSV = ATE_RUNS_DIR / f'seed_{SEED}' / 'test_predictions.csv'

if not ATE_CSV.is_file():
    raise FileNotFoundError(f'ATE prediction was not created: {ATE_CSV}')
print('mT5-large ATE prediction ready:', ATE_CSV)

In [ ]:
import torch
import common.run_multiseed as runner

# Register both APC backbones and exactly five resize configs.
runner.MODEL_REGISTRY.update(APC_MODELS)
runner.NUM_EPOCHS = MAX_EPOCHS
runner.TRAIN_APC = DATA_DIR / 'train.apc'
runner.DEV_APC = DATA_DIR / 'dev.apc'
runner.TEST_APC = DATA_DIR / 'test.apc'
runner.SUPPLEMENT_DIR = DATA_DIR / 'supplement'
runner.ALL_CONFIGS = [
    (True, True, True, True, 'bipartite', False, 'BiToMe+CDM', 'lcf_bip_cdm'),
    (True, True, True, True, 'sequential_local', False, 'SLM+CDM', 'lcf_seq_cdm'),
    (True, False, True, True, 'sequential_local', False, 'SLM+CDW', 'lcf_seq_cdw'),
    (True, True, True, True, 'sequential_cosine', False, 'SCM+CDM', 'lcf_scm_cdm'),
    (True, False, True, True, 'sequential_cosine', False, 'SCM+CDW', 'lcf_scm_cdw'),
]
runner.CONFIG_BY_ID = {config[7]: config for config in runner.ALL_CONFIGS}

args = [
    '--seeds', str(SEED),
    '--model-types', *APC_MODELS.keys(),
    '--configs', *METHODS.keys(),
    '--runs-dir', str(RUNS_DIR),
    '--resume',
]
if ATE_CSV.is_file():
    args += ['--ate-csv', str(ATE_CSV)]
print('Ready:', torch.cuda.is_available(), 'CUDA; output =', RUNS_DIR)
print('APC matrix:', len(APC_MODELS), 'models x', len(METHODS), 'methods =', len(APC_MODELS) * len(METHODS), 'runs')
print('Configs:', [config[7] for config in runner.ALL_CONFIGS])

In [ ]:
# Run 10 combinations. Re-running is safe because --resume skips completed rows.
sys.argv = ['run_multiseed.py', *args]
runner.main()

In [ ]:
import pandas as pd

raw = RUNS_DIR / 'results_raw.csv'
if raw.is_file():
    df = pd.read_csv(raw)
    print(f'Completed: {len(df)} / {len(METHODS) * len(APC_MODELS)}')
    metric_columns = [
        'model_type', 'config_id', 'seed',
        'joint_precision', 'joint_recall', 'joint_f1',
        'joint_precision_macro', 'joint_recall_macro', 'joint_f1_macro',
        'joint_acc',
        'e2e_micro_precision', 'e2e_micro_recall', 'e2e_micro_f1',
        'e2e_macro_precision', 'e2e_macro_recall', 'e2e_macro_f1',
    ]
    available = [column for column in metric_columns if column in df.columns]
    display(df[available].sort_values(['model_type', 'config_id']))
    evaluation_csv = RUNS_DIR / 'evaluation_precision_recall_f1.csv'
    df[available].to_csv(evaluation_csv, index=False)
    print('Evaluation table:', evaluation_csv)
else:
    print('No results yet:', raw)